# Mistral 7B Medical Model Evaluation

This notebook focuses only on evaluating the fine-tuned Mistral 7B model on the development dataset.

In [1]:
# Import required libraries
import os
import json
import torch
import re
import gc
from pathlib import Path
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Basic configurations
class Config:
    PROCESSED_DATA_DIR = Path("processed_data")
    DEV_PATH = PROCESSED_DATA_DIR / "processed_dev.json"
    MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
    MODEL_OUTPUT_DIR = Path("model_output")

config = Config()

In [2]:
# Load development data
def load_dev_data():
    try:
        if not config.DEV_PATH.exists():
            print(f"File not found: {config.DEV_PATH}")
            return []
        
        with open(config.DEV_PATH, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"Successfully loaded {len(data)} development examples")
        return data
    except Exception as e:
        print(f"Error loading data: {str(e)}")
        return []

dev_data = load_dev_data()
if not dev_data:
    print("No development data available. Cannot perform evaluation.")

Successfully loaded 60 development examples


In [3]:
def clean_response(text):
    """Clean the response by removing repetitive text"""
    # Remove unwanted phrases
    if "would you like to video or text chat with me" in text.lower():
        idx = text.lower().find("would you like to video or text chat with me")
        if idx > 0:
            text = text[:idx].strip()
    
    patterns = ["godspeed", "just click the button below", 
                "i can answer your questions now", "(\\d+/\\d+/\\d+)"]
    
    for pattern in patterns:
        text = re.sub(f"(?i){pattern}", "", text)
    
    # Clean up newlines
    text = re.sub(r'\n\s*\n', '\n\n', text)
    return text.strip()

def generate_response(query, model_path=None):
    """Generate a response using the model"""
    try:
        # Set model path
        if model_path is None:
            model_path = str(config.MODEL_OUTPUT_DIR)
        
        # Load tokenizer and model
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        
        # Configure quantization
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
        )
        
        # Load base model
        base_model = AutoModelForCausalLM.from_pretrained(
            config.MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
        
        # Load adapter
        model = PeftModel.from_pretrained(base_model, model_path)
        
        # Prepare input
        input_text = f"Patient: {query}\nDoctor:"
        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
        
        # Generate output
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=512,
                temperature=0.7,
                do_sample=True,
                repetition_penalty=1.2,
                no_repeat_ngram_size=3
            )
        
        # Process output
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract response
        response_start = generated_text.find("Doctor:")
        if response_start != -1:
            response = generated_text[response_start + len("Doctor:"):].strip()
        else:
            response = generated_text[len(input_text):].strip()
        
        return clean_response(response)
    
    finally:
        # Clean up memory
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

In [ ]:
def evaluate_dev_set(num_samples=3, model_path=None):
    """Evaluate model on development set samples"""
    if not dev_data:
        print("No development data available for evaluation.")
        return
    
    # Select random samples
    import random
    samples = random.sample(dev_data, min(num_samples, len(dev_data)))
    
    print(f"\n=== Evaluating Model on {len(samples)} Dev Samples ===\n")
    
    for i, sample in enumerate(samples, 1):
        query = sample.get("patient_query", "")
        if not query:
            continue
            
        print(f"Sample {i}: {query[:100]}...")
        
        try:
            # Get model response
            generated_response = generate_response(query, model_path)
            
            print("\nGenerated Response:")
            print(f"{'-' * 40}")
            print(generated_response[:500]) # Limit output length for readability
            if len(generated_response) > 500:
                print("... (response truncated)")
            print(f"{'-' * 40}\n")
            
            # Force memory cleanup between samples
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
            
        except Exception as e:
            print(f"Error: {str(e)}")
            
        print("-" * 80 + "\n")

# Run the evaluation
evaluate_dev_set(num_samples=3)


=== Evaluating Model on 3 Dev Samples ===

Sample 1: hi dr. my whole body feels sore as well as my neck. my brain feels loose in my head and have headach...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Generated Response:
----------------------------------------
in brief: systemic infection you may have a systemic infectious process such as the flu or mono. check your temperature, if persistently elevated, call your md for instructions. i recommend also being tested for covid-19, depending on your location and recent travels/contacts.other possibilities include dehydration (especially if vomiting or diarrhea), lack of sleep, meningitis (uncommon).
----------------------------------------

--------------------------------------------------------------------------------

Sample 2: i have chest congestion and i feel heavy when i breathe (not sure if i have a fever, don’t really ha...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Generated Response:
----------------------------------------
stay home. stay at home, rest, drink fluids and monitor your temperature. you may have an infection other than covid-19, however given the situation with your boyfriend and the unknown status of his colleagues you should assume you are at risk for acquiring covid 19 and avoid contact with others, wear a mask and consider testing if symptomatic or if you have underlying medical conditions that put you at higher risk for complications. . . . () read https://www.healthtap.com/blog/covid-care-guidel
... (response truncated)
----------------------------------------

--------------------------------------------------------------------------------

Sample 3: i am the main member of discovery, i want to complete a health covid assessment for my husband? plea...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
